# 02 - EDA: LMP, generation, and revenue for DPC.FLAMBEAU

**Goal.** Understand the seasonal, weekly, and diurnal patterns of LMP and generation, and see how they combine into revenue. The recommendation notebook (03) picks its 2-week windows from this understanding.

**Load only.** Everything here reads from `data/processed/lmp_hourly.parquet` and `data/raw/flambeau_gen.csv`; no network I/O.

**Central question.** LMP has a summer peak (AC load) and generation has a spring peak (snowmelt) - are they anti-correlated seasonally? If yes, revenue seasonality is flatter than either input, and the optimal window is where their product is smallest, not where either factor is smallest.

In [ ]:
# Step 1. Imports, plot styling, paths.
#   - matplotlib inline, sensible figure size.
#   - PROJECT_ROOT = Path(...).parent.parent; add to sys.path.
#   - LMP_PATH = PROJECT_ROOT / "data" / "processed" / "lmp_hourly.parquet"
#   - GEN_PATH = PROJECT_ROOT / "data" / "raw" / "flambeau_gen.csv"


In [ ]:
# Step 2. Load.
#   lmp = pd.read_parquet(LMP_PATH)                       # hourly UTC index, col 'lmp'
#   gen_raw = load_generation_csv(GEN_PATH)                # 4-day cadence, col 'mw'
#   gen = trim_to_usable_range(gen_raw)                    # drop 2015 NaN block
#
# Print shape and index bounds of each so you catch a range mismatch
# before it silently truncates downstream joins.


## 2a. LMP patterns

We want to know: annual level drift, seasonality, weekly shape, diurnal shape, and how heavy the tail is.

In [ ]:
# Step 3. Annual mean LMP.
#   - Groupby year, mean.
#   - Bar chart. 2022 should visibly dwarf every other year.
#   - This is the picture that motivates 'rank by share, not dollars' in
#     the windows notebook.


In [ ]:
# Step 4. Monthly climatology.
#   - Groupby (year, month) mean, then average across years -> 12 numbers.
#   - Overlay a spread band (10-90 percentile across years) to show how
#     stable the seasonal shape is.
#   - Expected: summer peak (Jun-Aug), shoulder trough in spring and fall.


In [ ]:
# Step 5. Hour-of-day and day-of-week shape.
#   - Convert index to Central time first (CENTRAL_TZ from src.gen).
#   - Groupby local hour, mean.
#   - Groupby local dayofweek, mean.
#   - These plots motivate the caveat about flat-within-day generation
#     erasing any price-following behavior.


In [ ]:
# Step 6. Tail behavior.
#   - Fraction of hours with LMP < 0, per month.
#   - 99th, 99.9th percentile per year - shows how outlier-driven any
#     'mean revenue' number is.


## 2b. Generation patterns

In [ ]:
# Step 7. Monthly climatology of MW.
#   - Same shape as Step 4 but on gen['mw'].
#   - Expected: April-May snowmelt peak, late-summer / autumn minimum.


In [ ]:
# Step 8. Annual mean MW.
#   - Is there a multi-year trend? Anything to flag in the writeup?
#   - Also plot the empirical CDF of MW; a hard ceiling near ~23.7 MW is
#     evidence of capacity-clipping, worth mentioning.


## 2c. Revenue - the product that actually matters

This is where the window recommendation lives. If LMP and MW are anti-seasonal, the seasonal shape of `MW * LMP` will be flatter than either factor, and the answer must come from the product.

In [ ]:
# Step 9. Build hourly and daily revenue.
#   gen_hourly = interpolate_hourly(gen)
#   hourly = build_hourly_revenue(gen_hourly, lmp)
#   daily = build_daily_revenue(hourly)
#
# Persist daily to data/processed/daily_revenue.parquet so notebook 03
# does not have to redo the interpolation.


In [ ]:
# Step 10. Monthly climatology of daily revenue.
#   - Groupby local month, mean of daily revenue.
#   - Overlay a spread band across years.
#   - This is one of the three headline charts for the presentation.


In [ ]:
# Step 11. 14-day rolling sum of daily revenue, plotted per year on a
#          common (month, day) x-axis so seasonality is visible across
#          years and inter-annual variation shows up as spread.
#
# This is the second headline chart. The final window recommendation
# is essentially the argmin of the *mean* of these traces after
# normalization.


In [ ]:
# Step 12. Sensitivity to the interpolation choice.
#   - Rebuild `daily` using `mw.ffill()` instead of time interpolation.
#   - Compare 14-day rolling sums: percent difference per date.
#   - If the max abs percent difference is small (a couple percent),
#     the interpolation choice is not what drives the answer - flag
#     that in the writeup.
